## Interactive CTD depth-bin boxplots

This notebook reads the summary stats CSV in this folder and provides dropdowns to choose:
- `variable`
- `station_type`
- `month`

It then plots a box-style summary by `depth_bin_low` (x-axis) using:
- **median** from `median`
- **box lower/upper** from `p01` / `p99`
- **whiskers** from `min` / `max`


In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

if 'notebook_connected' in pio.renderers:
    pio.renderers.default = 'notebook_connected'

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception as e:
    raise RuntimeError(
        "ipywidgets is required for the dropdown UI. Install with: pip install ipywidgets"
    ) from e

# If you have multiple CSVs in this folder, this picks the first one.
csvs = sorted(Path('.').glob('*.csv'))
if not csvs:
    raise FileNotFoundError('No .csv files found in the current folder.')

csv_path = csvs[0]
print(f"Reading: {csv_path}")

df = pd.read_csv(csv_path)

required_cols = {
    'variable',
    'station_type',
    'month',
    'depth_bin_low',
    'depth_bin_high',
    'n',
    'median',
    'p01',
    'p99',
    'min',
    'max',
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

# Normalize types.
df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
df['depth_bin_low'] = pd.to_numeric(df['depth_bin_low'], errors='coerce')
df['depth_bin_high'] = pd.to_numeric(df['depth_bin_high'], errors='coerce')
for c in ['median', 'p01', 'p99', 'min', 'max', 'n']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=['variable', 'station_type', 'month', 'depth_bin_low', 'median', 'p01', 'p99', 'min', 'max'])

variables = sorted(df['variable'].unique().tolist())
station_types = sorted(df['station_type'].unique().tolist())
months = sorted(df['month'].dropna().unique().astype(int).tolist())

variables[:5], station_types[:5], months[:12]

Reading: stats_by_depth_all.csv


(['conductivity',
  'oxygen_dissolved_oxygen',
  'practical_salinity',
  'pressure',
  'temperature'],
 ['deep_cast', 'shallow_cast', 'shallow_stable'],
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])

In [2]:
def _subset(var: str, station_type: str, month: int) -> pd.DataFrame:
    sub = df[(df['variable'] == var) & (df['station_type'] == station_type) & (df['month'] == month)].copy()
    sub = sub.sort_values(['depth_bin_low', 'depth_bin_high'])
    return sub


def _depth_low_labels(sub: pd.DataFrame) -> list[str]:
    # Use a categorical x-axis so only the available depth bins appear (no gaps).
    return [f"{v:g}" for v in sub['depth_bin_low'].to_list()]


def _hover_text(sub: pd.DataFrame) -> list[str]:
    parts = zip(
        sub['depth_bin_low'],
        sub['depth_bin_high'],
        sub['n'],
        sub['min'],
        sub['p01'],
        sub['median'],
        sub['p99'],
        sub['max'],
    )
    return [
        (
            f"Depth bin: {low:g}–{high:g}<br>"
            f"n={n:.0f}<br>"
            f"min={mn:.4g}<br>"
            f"p01={p01:.4g}<br>"
            f"median={med:.4g}<br>"
            f"p99={p99:.4g}<br>"
            f"max={mx:.4g}"
        )
        for low, high, n, mn, p01, med, p99, mx in parts
    ]


def _build_trace(sub: pd.DataFrame) -> go.Box:
    # Plotly supports precomputed box stats via q1/median/q3 and whisker fences.
    # Here we map p01->q1 and p99->q3 per your request.
    return go.Box(
        x=_depth_low_labels(sub),
        q1=sub['p01'],
        median=sub['median'],
        q3=sub['p99'],
        lowerfence=sub['min'],
        upperfence=sub['max'],
        boxpoints=False,
        text=_hover_text(sub),
        hovertemplate="%{text}<extra></extra>",
    )


def _title(var: str, station_type: str, month: int) -> str:
    return f"{var} | {station_type} | month={month}"


# Initial selections
init_var = variables[0]
init_station = station_types[0]
init_month = months[0]

sub0 = _subset(init_var, init_station, init_month)
if sub0.empty:
    raise RuntimeError("Initial filter produced no rows; check the CSV contents.")

fig = go.FigureWidget(data=[_build_trace(sub0)])
fig.update_layout(
    title=_title(init_var, init_station, init_month),
    xaxis_title='depth_bin_low',
    xaxis=dict(type='category', categoryorder='array', categoryarray=_depth_low_labels(sub0)),
    yaxis_title=init_var,
    template='plotly_white',
    height=550,
    margin=dict(l=60, r=30, t=60, b=60),
)

var_dd = widgets.Dropdown(options=variables, value=init_var, description='variable:')
station_dd = widgets.Dropdown(options=station_types, value=init_station, description='station_type:')
month_dd = widgets.Dropdown(options=months, value=init_month, description='month:')

status = widgets.HTML(value='')


def _update(*_):
    v = var_dd.value
    s = station_dd.value
    m = int(month_dd.value)

    sub = _subset(v, s, m)
    if sub.empty:
        status.value = '<b>No data for this selection.</b>'
        with fig.batch_update():
            fig.data[0].x = []
            fig.data[0].q1 = []
            fig.data[0].median = []
            fig.data[0].q3 = []
            fig.data[0].lowerfence = []
            fig.data[0].upperfence = []
            fig.data[0].text = []
            fig.layout.xaxis.categoryarray = []
            fig.layout.title = _title(v, s, m) + ' (no data)'
            fig.layout.yaxis.title = v
        return

    status.value = ''
    xlabels = _depth_low_labels(sub)
    with fig.batch_update():
        fig.data[0].x = xlabels
        fig.data[0].q1 = sub['p01']
        fig.data[0].median = sub['median']
        fig.data[0].q3 = sub['p99']
        fig.data[0].lowerfence = sub['min']
        fig.data[0].upperfence = sub['max']
        fig.data[0].text = _hover_text(sub)
        fig.layout.xaxis.categoryarray = xlabels
        fig.layout.title = _title(v, s, m)
        fig.layout.yaxis.title = v


for w in (var_dd, station_dd, month_dd):
    w.observe(_update, names='value')

controls = widgets.HBox([var_dd, station_dd, month_dd])
display(widgets.VBox([controls, status, fig]))

# Trigger once in case widgets default differ
_update()

In [3]:
# --- Plot 2: Month-on-x box summaries for a chosen depth bin ---

def _depth_bin_options_for(var: str, station_type: str) -> list[tuple[str, tuple[float, float]]]:
    bins = (
        df[(df['variable'] == var) & (df['station_type'] == station_type)][['depth_bin_low', 'depth_bin_high']]
        .drop_duplicates()
        .dropna()
        .sort_values(['depth_bin_low', 'depth_bin_high'])
    )

    opts: list[tuple[str, tuple[float, float]]] = []
    for low, high in bins[['depth_bin_low', 'depth_bin_high']].itertuples(index=False, name=None):
        low_f = float(low)
        high_f = float(high)
        opts.append((f"{low_f:g}–{high_f:g}", (low_f, high_f)))
    return opts


depth_bin_options = _depth_bin_options_for(variables[0], station_types[0])
if not depth_bin_options:
    raise RuntimeError('No depth bins found for the initial variable/station_type selection.')


def _subset_by_depth(var: str, station_type: str, depth_bin: tuple[float, float]) -> pd.DataFrame:
    low, high = depth_bin
    sub = df[
        (df['variable'] == var)
        & (df['station_type'] == station_type)
        & (df['depth_bin_low'] == low)
        & (df['depth_bin_high'] == high)
    ].copy()
    sub = sub.sort_values(['month'])
    return sub


def _hover_text_month(sub: pd.DataFrame) -> list[str]:
    parts = zip(
        sub['month'].astype(int),
        sub['n'],
        sub['min'],
        sub['p01'],
        sub['median'],
        sub['p99'],
        sub['max'],
    )
    return [
        (
            f"Month: {m}<br>"
            f"n={n:.0f}<br>"
            f"min={mn:.4g}<br>"
            f"p01={p01:.4g}<br>"
            f"median={med:.4g}<br>"
            f"p99={p99:.4g}<br>"
            f"max={mx:.4g}"
        )
        for m, n, mn, p01, med, p99, mx in parts
    ]


def _build_trace_month(sub: pd.DataFrame) -> go.Box:
    return go.Box(
        x=sub['month'].astype(int),
        q1=sub['p01'],
        median=sub['median'],
        q3=sub['p99'],
        lowerfence=sub['min'],
        upperfence=sub['max'],
        boxpoints=False,
        text=_hover_text_month(sub),
        hovertemplate="%{text}<extra></extra>",
    )


def _title_month(var: str, station_type: str, depth_bin: tuple[float, float]) -> str:
    low, high = depth_bin
    return f"{var} | {station_type} | depth={low:g}–{high:g}"


init_var2 = variables[0]
init_station2 = station_types[0]
# depth_bin_options depends on variable+station_type
init_depth2 = depth_bin_options[0][1]

subm0 = _subset_by_depth(init_var2, init_station2, init_depth2)

# Keep a single trace and update its arrays for widget reliability.
empty_trace = go.Box(
    x=[],
    q1=[],
    median=[],
    q3=[],
    lowerfence=[],
    upperfence=[],
    boxpoints=False,
    text=[],
    hovertemplate="%{text}<extra></extra>",
)

trace0 = _build_trace_month(subm0) if not subm0.empty else empty_trace

fig2 = go.FigureWidget(data=[trace0])
fig2.update_layout(
    title=_title_month(init_var2, init_station2, init_depth2) + ("" if not subm0.empty else " (no data)"),
    xaxis_title='month',
    xaxis=dict(type='category', categoryorder='array', categoryarray=months),
    yaxis_title=init_var2,
    template='plotly_white',
    height=550,
    margin=dict(l=60, r=30, t=60, b=60),
)

var2_dd = widgets.Dropdown(options=variables, value=init_var2, description='variable:')
station2_dd = widgets.Dropdown(options=station_types, value=init_station2, description='station_type:')
depth2_dd = widgets.Dropdown(options=depth_bin_options, value=init_depth2, description='depth_bin:')

status2 = widgets.HTML(value='')


def _sync_depth_options():
    v = var2_dd.value
    s = station2_dd.value
    opts = _depth_bin_options_for(v, s)
    if not opts:
        # Keep current options; we'll show "no data" below.
        return

    current = depth2_dd.value
    with depth2_dd.hold_trait_notifications():
        depth2_dd.options = opts
        if current not in [o[1] for o in opts]:
            depth2_dd.value = opts[0][1]


def _update2(*_):
    # Ensure depth-bin choices match current variable/station_type.
    _sync_depth_options()

    v = var2_dd.value
    s = station2_dd.value
    d = depth2_dd.value

    sub = _subset_by_depth(v, s, d)

    if sub.empty:
        status2.value = '<b>No data for this selection.</b>'
        with fig2.batch_update():
            fig2.data[0].x = []
            fig2.data[0].q1 = []
            fig2.data[0].median = []
            fig2.data[0].q3 = []
            fig2.data[0].lowerfence = []
            fig2.data[0].upperfence = []
            fig2.data[0].text = []
            fig2.layout.title = _title_month(v, s, d) + ' (no data)'
            fig2.layout.yaxis.title = v
        return

    status2.value = ''
    with fig2.batch_update():
        fig2.data[0].x = sub['month'].astype(int)
        fig2.data[0].q1 = sub['p01']
        fig2.data[0].median = sub['median']
        fig2.data[0].q3 = sub['p99']
        fig2.data[0].lowerfence = sub['min']
        fig2.data[0].upperfence = sub['max']
        fig2.data[0].text = _hover_text_month(sub)
        fig2.layout.title = _title_month(v, s, d)
        fig2.layout.yaxis.title = v


for w in (var2_dd, station2_dd, depth2_dd):
    w.observe(_update2, names='value')

controls2 = widgets.HBox([var2_dd, station2_dd, depth2_dd])
display(widgets.VBox([controls2, status2, fig2]))

_update2()